In [11]:
import pandas as pd
import numpy as np  
from scipy.stats import norm  
import matplotlib.pyplot as plt

The simulation used $\lambda$ = 0.000005 and a window size of 300.

In [27]:
simulation_data_lasso = pd.read_csv(r'C:\Users\jonat\Lasso_paper\Theory\simulation_data_lasso.csv')
simulation_data_ols = pd.read_csv(r'C:\Users\jonat\Lasso_paper\Theory\simulation_data_ols_soft.csv')

In [28]:
print(simulation_data_ols.columns)

Index(['Unnamed: 0', 'x_1', 'r', 'beta_soft1', 'beta_ols1'], dtype='object')


In [13]:
# calculate the probability with which beta is selected in each period
beta_1_count_lasso = (simulation_data_lasso['beta_1'] != 0).sum()
selection_prob_beta1_lasso = beta_1_count_lasso / len(simulation_data_lasso)

print(f"Selection probability beta1 Lasso: {selection_prob_beta1_lasso}")

Selection probability beta1 Lasso: 0.0268


In [79]:
# Parameter calibration
a = 1.001 # mean of the dividend process
gamma = 2.0 # risk aversion
sigma = 0.01 # std dev of the dividend process
sigma_x = 0.01 # std dev of the observable
delta = 0.96 # discount factor
phi = np.exp(sigma**2 / 2) # correction term for lognormality
kappa = delta * a ** ( - gamma) * phi
w = - np.log(kappa) - np.exp(-10)  # small constant to ensure positivity


# simulation parameters
lam = 0.00001 # alpha used in scikit-learn
window = 300 # rolling window size
p = 0.9999999999999 # weight used in approximation of gain parameter
sigma_beta = simulation_data_ols['beta_ols1'].std() # std dev of the coefficient beta_1 from ols with soft-thresholding

In [80]:
# what gamma corresponds to our window size?
def implied_g(window, p):
    return 1.0 - (1 - p) ** (1 / window)

g = implied_g(window, p)
print(g)

0.0949613699365649


In [82]:
# calculate sigma_ols
sigma_ols = np.sqrt(g * sigma ** 2  / (2.0 * sigma_x ** 2 * (1.0 + kappa / (1.0 - kappa))))

# adjust lambda for the scaling in scikit-learn
lam_adjusted = lam / (sigma_x ** 2)

# calculate z
z = lam_adjusted / sigma_ols

# equation from proposition 3
p_nonzero = 2 * (1 - norm.cdf(z))

print(f"Theoretically implied selection probability: {p_nonzero}")
print(f"Simulated selection probability: {selection_prob_beta1_lasso}")

Theoretically implied selection probability: 0.024908535941275733
Simulated selection probability: 0.0268


In [83]:
print(sigma_ols), print(sigma_beta)

0.044586760065865504
0.04721821804152477


(None, None)

In [ ]:
# # counterfactual: we take the beta_ols estimates and the lambda that was used for soft-threshlolding in simulation
# # average value of se_xx 
# se_xx = 0.05

# # take moments from simulation_data_ols
# se_ols = simulation_data_ols['beta_1'].std() 
# se_x = simulation_data_ols['x_1'].std()

# # compute lambda 
# lam_raw = 0.000005
# lam_adjusted = (lam_raw / se_xx) * 300

# # other values from calibration
# a = 1.001 # mean of the dividend process
# gamma = 2.0 # risk aversion
# sigma = 0.01 # std dev of the dividend process
# delta = 0.96 # discount factor
# phi = np.exp(sigma**2 / 2) # correction term for lognormality
# kappa = delta * a ** ( - gamma) * phi
# w = - np.log(kappa) - np.exp(-10)  # small constant to ensure positivity


# z = lam_adjusted / se_ols

# p_nonzero = 1 - (norm.cdf(z) - norm.cdf(-z))   

# print(f"Theoretically implied selection probability: {p_nonzero}")


Theoretically implied selection probability: 3.681943638866869e-12
